# 🚀 Fase 9: QLoRA Fine-Tuning pada Qwen2.5-1.5B-Instruct
### **Temporal Regulatory Question Answering System (OJK & Bank Indonesia)**

Notebook ini mengeksekusi proses **Fine-Tuning Parameter-Efficient (QLoRA 4-bit)** pada model dasar `Qwen2.5-1.5B-Instruct` menggunakan GPU **Nvidia T4** di Google Colab.

> **Instruksi Runtime:** Pastikan runtime diset ke **T4 GPU** sebelum menjalankan (`Runtime` -> `Change runtime type` -> `T4 GPU`).

In [ ]:
# 1. Verifikasi Ketersediaan GPU Nvidia T4
!nvidia-smi

In [ ]:
# 2. Clone Repositori Riset dari GitHub
!git clone https://github.com/T-Rex2586/RAG-Regulasi-Temporal-untuk-Regulasi-OJK-Bank-Indonesia.git
%cd RAG-Regulasi-Temporal-untuk-Regulasi-OJK-Bank-Indonesia

In [ ]:
# 3. Instalasi Dependensi QLoRA (HuggingFace, PEFT, BitsAndBytes)
!pip install -q transformers>=4.40.0 peft>=0.10.0 bitsandbytes>=0.43.0 accelerate>=0.28.0 datasets pyyaml matplotlib
!pip uninstall -y torchao


In [ ]:
# 4. Verifikasi Anti-Kebocoran & Generate Dataset Instruksi Latihan
# Dataset dibuat secara murni dari train.jsonl (75 kueri) tanpa menyentuh dev/test
!python -m src.train.prepare_dataset

In [ ]:
# 5. Jalankan Pelatihan QLoRA 4-bit (NF4, Double Quant, r=16, alpha=32)
# Waktu estimasi: ~12-18 menit pada Colab T4 GPU
!python -m src.train.train_qlora --config configs/qlora.yaml

In [ ]:
# 6. Visualisasi & Plot Kurva Training Loss
import json
import matplotlib.pyplot as plt

loss_file = 'models/qlora_adapter_b5/training_loss.json'
with open(loss_file, 'r', encoding='utf-8') as f:
    history = json.load(f)

steps = [h['step'] for h in history if h.get('loss') is not None]
losses = [h['loss'] for h in history if h.get('loss') is not None]

plt.figure(figsize=(9, 5))
plt.plot(steps, losses, marker='o', color='#1E3A8A', linewidth=2, label='Training Loss')
plt.title('QLoRA Training Loss Curve: Qwen2.5-1.5B (Fase 9)', fontsize=14, fontweight='bold')
plt.xlabel('Training Steps', fontsize=12)
plt.ylabel('Cross-Entropy Loss', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('models/qlora_adapter_b5/loss_curve.png', dpi=300)
plt.show()
print('Kurva loss tersimpan -> models/qlora_adapter_b5/loss_curve.png')

In [ ]:
# 7. Pengujian Kualitatif Model: Base vs Adapted (In-Domain & OOD)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_id = 'Qwen/Qwen2.5-1.5B-Instruct'
adapter_dir = 'models/qlora_adapter_b5'

tokenizer = AutoTokenizer.from_pretrained(base_id)
model = AutoModelForCausalLM.from_pretrained(base_id, torch_dtype=torch.float16, device_map='auto')
model = PeftModel.from_pretrained(model, adapter_dir)
model.eval()

test_prompt = [
    {'role': 'system', 'content': 'Anda adalah asisten kepatuhan hukum perbankan OJK dan Bank Indonesia.'},
    {'role': 'user', 'content': 'EVIDENCE:\n[POJK No. 12/2021] [Pasal 2] Bank wajib memiliki modal disetor minimal 3 Triliun Rupiah.\n\nPERTANYAAN:\nBerapa modal disetor minimal bank umum?\nRESPONS JSON:'}
]

inputs = tokenizer.apply_chat_template(test_prompt, tokenize=True, return_tensors='pt', add_generation_prompt=True, return_dict=True).to('cuda')
input_ids = inputs['input_ids']
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=150, do_sample=False)
print('Output Teradaptasi QLoRA:')
print(tokenizer.decode(out[0][input_ids.shape[1]:], skip_special_tokens=True))


In [ ]:
# 8. Zip Adapter Artifacts & Download ke Komputer Lokal
!zip -r qlora_adapter_b5.zip models/qlora_adapter_b5

from google.colab import files
files.download('qlora_adapter_b5.zip')
print('Download file qlora_adapter_b5.zip selesai. Ekstrak ke folder models/qlora_adapter_b5 di laptop Anda.')